In [18]:
import pandas as pd 
import numpy as np

In [19]:
ENDERECO_DADOS = 'https://www.ispdados.rj.gov.br/Arquivos/BaseDPEvolucaoMensalCisp.csv'

df_roubos = pd.read_csv(ENDERECO_DADOS, sep=';', encoding='iso-8859-1')
print(df_roubos.head())
# print(df_roubos.describe())
# print(df_roubos['regiao'].unique())


   cisp  mes   ano  mes_ano  aisp  risp           munic    mcirc   regiao  \
0     1    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
1     4    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
2     5    1  2003  2003m01     5     1  Rio de Janeiro  3304557  Capital   
3     6    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   
4     7    1  2003  2003m01     1     1  Rio de Janeiro  3304557  Capital   

   hom_doloso  ...  cmp  cmba  ameaca  pessoas_desaparecidas  \
0           0  ...  NaN   NaN      21                      2   
1           3  ...  NaN   NaN      15                      6   
2           3  ...  NaN   NaN      47                      2   
3           6  ...  NaN   NaN      26                      2   
4           4  ...  NaN   NaN      10                      1   

   encontro_cadaver  encontro_ossada  pol_militares_mortos_serv  \
0                 0                0                          0   
1                 

In [20]:
# Delimitar as variáveis cisp regiao munic roubo_carga
df_roubos = df_roubos[['cisp', 'regiao', 'munic', 'roubo_carga']]

# Corrigindo a Grande Niterói
correcao_acentuacao = df_roubos['regiao'].str.startswith('Grande Niter', na=False)
# print(correcao_acentuacao)
df_roubos.loc[correcao_acentuacao, 'regiao'] = 'Grande Niterói'
print(df_roubos['regiao'].unique())

<StringArray>
['Capital', 'Baixada Fluminense', 'Interior', 'Grande Niterói']
Length: 4, dtype: str


In [21]:
# Agrupando e totalizando os roubos de carga por cisp regiao e munic
df_roubos = df_roubos.groupby(['cisp', 'regiao', 'munic'], as_index=False)['roubo_carga'].sum()
display(df_roubos)

,cisp,regiao,munic,roubo_carga
0,1,Capital,Rio de Janeiro,810
1,4,Capital,Rio de Janeiro,477
2,5,Capital,Rio de Janeiro,640
3,6,Capital,Rio de Janeiro,887
4,7,Capital,Rio de Janeiro,149
...,...,...,...,...
142,165,Interior,Mangaratiba,51
143,166,Interior,Angra dos Reis,211
144,167,Capital,Paraty,0
145,167,Interior,Paraty,6


In [22]:
# Calculando média, mediana, total, maior, menor e quartis
array_roubos = np.array(df_roubos['roubo_carga'])

media = np.mean(array_roubos)
mediana = np.median(array_roubos)

maximo = np.max(array_roubos)
minimo = np.min(array_roubos)
total = np.sum(array_roubos)


q1 = np.percentile(array_roubos, 25)
q3 = np.percentile(array_roubos, 75)

print('Média: ', media)
print('Mediana: ', mediana)
print('Máximo: ', maximo)
print('Mínimo: ', minimo)
print('Total: ', total)

print('Quartil 1: ', q1)
print('Quartil 3: ', q3)

Média:  796.9251700680272
Mediana:  207.0
Máximo:  8732
Mínimo:  0
Total:  117148
Quartil 1:  46.0
Quartil 3:  842.0


In [23]:
# Criando uma série com os maiores
df_maiores = df_roubos[df_roubos['roubo_carga'] >= q3].copy()
df_maiores['flag'] = 'mais'
display(df_maiores)

# Criando uma série com os menores
df_menores = df_roubos[df_roubos['roubo_carga'] <= q1].copy()
df_menores['flag'] = 'menos'
display(df_menores)



,cisp,regiao,munic,roubo_carga,flag
3,6,Capital,Rio de Janeiro,887,mais
13,17,Capital,Rio de Janeiro,1766,mais
16,20,Capital,Rio de Janeiro,853,mais
17,21,Capital,Rio de Janeiro,5134,mais
18,22,Capital,Rio de Janeiro,3758,mais
19,23,Capital,Rio de Janeiro,987,mais
20,24,Capital,Rio de Janeiro,1212,mais
21,25,Capital,Rio de Janeiro,2048,mais
22,26,Capital,Rio de Janeiro,883,mais
23,27,Capital,Rio de Janeiro,2297,mais


,cisp,regiao,munic,roubo_carga,flag
7,11,Capital,Rio de Janeiro,20,menos
79,91,Interior,Valença,26,menos
80,92,Interior,Rio das Flores,5,menos
83,94,Interior,Piraí;Pinheiral,41,menos
85,96,Interior,Miguel Pereira;Paty dos Alferes,6,menos
86,97,Interior,Mendes,4,menos
87,98,Interior,Engenheiro Paulo de Frontin,7,menos
89,100,Interior,Porto Real;Quatis,46,menos
90,101,Interior,Pinheiral,8,menos
91,104,Interior,São José do Vale do Rio Preto,9,menos


In [24]:
# Concatenando os dois dataframes df_maiores e df_menores
df_roubos_flags = pd.concat([df_maiores, df_menores], ignore_index=True)

In [25]:
# exportando os dados utf-8-sig p/ ler no powerbi
df_roubos_flags.to_csv('roubos_flags.csv', index=False, encoding='utf-8-sig')